# Demo end-to-end - MMM dalla A alla Z

Notebook completo per la **dimostrazione**: esegue l'intera pipeline su dati sintetici, tutto qui su Colab, **senza preparazione in locale e senza upload di file**.

**Flusso:**
1. **Setup** - codice + dipendenze
2. **Generazione** - crea dati finti realistici (~2 anni, 20 regioni, 4 canali + CRM) con una *verita' nascosta* (ROI e adstock reali)
3. **Ingestion** - legge gli export grezzi, riconosce le colonne, pseudonimizza (GDPR) e aggrega -> 4 file canonici
4. **Fit** - Meridian stima ROI, adstock e curve di risposta (serve GPU)
5. **Test (parameter recovery)** - confronta le stime con la verita' nascosta: se coincidono, il metodo funziona

**Prima di iniziare:** `Runtime -> Cambia tipo di runtime -> GPU (T4) -> Salva`.

## 1. Setup: codice e dipendenze

Clona il repo e installa le dipendenze. Durante l'installazione Colab puo' mostrare righe `ERROR: pip's dependency resolver ...` su pacchetti suoi (es. gradio): sono **innocue**, non riguardano la pipeline.

In [ ]:
!git clone https://github.com/Giacomod2001/Tesi-MMM.git
%cd Tesi-MMM
!pip install -q -r pipeline/requirements.txt

## 2. Genera i dati sintetici (con verita' nota)

Simula ~2 anni di attivita' su 20 regioni e 4 canali (Google, Meta, LinkedIn, Indeed) + CRM candidature. Produce export "sporchi" in stile piattaforma e, separatamente, `ground_truth.json` con i parametri veri (**il modello non lo legge mai**). I **ROI veri** stampati qui sotto sono cio' che il modello dovra' ritrovare.

In [ ]:
!python -m pipeline.generator.run

## 3. Ingestion: dagli export grezzi ai dati puliti

Il sistema legge i file grezzi, **riconosce automaticamente** le colonne (mappatura semi-automatica sul vocabolario canonico), applica la pseudonimizzazione GDPR e l'aggregazione regione x settimana, e scrive i **4 file canonici** (media, demand, seasonality, outcome).

> Nell'app reale un operatore **conferma** la mappatura (human-in-the-middle). Qui la confermiamo in automatico per scorrere la demo; sotto vedi la mappatura proposta e la **verifica** che l'ingestion riproduce fedelmente la verita'.

In [ ]:
# Ingestion programmatica: propone la mappatura, la auto-conferma, costruisce i fatti canonici
import os
import pandas as pd
from pipeline.ingestion import build
from pipeline import config

proposed, tables = build.propose_plan(config.RAW_DIR)
print("MAPPATURA PROPOSTA (auto-confermata per la demo)\n")
for p in proposed:
    canale = f"  [canale: {p.channel}]" if p.channel else ""
    print(f"- {p.file}  ->  {p.kind}{canale}")
    for c in p.columns:
        print(f"      {c.source!r:<32} -> {c.field}  ({c.confidence:.0%})")
    p.confirmed = True

res = build.ingest(config.RAW_DIR, plan=proposed, interactive=False, tables=tables)

print("\nFATTI CANONICI PRODOTTI:")
for name, df in res["facts"].items():
    print(f"  {name:<12} {len(df):>8,} righe")

# Verifica di fedelta': ingestion vs verita' (canonical_true, prodotto dal generatore)
tdir = os.path.join(config.DATA_DIR, "canonical_true")
sp_i = res["facts"]["media"].groupby("channel")["spend"].sum()
sp_t = pd.read_csv(os.path.join(tdir, "media.csv")).groupby("channel")["spend"].sum()
chk = pd.DataFrame({"spesa_ingestion": sp_i, "spesa_vera": sp_t})
chk["diff_%"] = (chk["spesa_ingestion"] / chk["spesa_vera"] - 1) * 100
print("\nVERIFICA fedelta' ingestion (spesa per canale):")
print(chk.round(2).to_string())
if "outcome" in res["facts"]:
    oi = res["facts"]["outcome"]
    ot = pd.read_csv(os.path.join(tdir, "outcome.csv"))
    print(f"\nConversioni: ingestion={oi['conversions'].sum():,.0f}  vere={ot['conversions'].sum():,.0f}")

## 4. Fit del modello (Meridian - serve GPU, ~20-40 min)

Il cuore del sistema: un MMM bayesiano geo-gerarchico stima ROI, adstock e curve di risposta per canale, con prior calibrati sul ROAS di piattaforma.

> Demo dal vivo: il fit dura ~20-40 min. Conviene avviarlo e usare l'attesa per spiegare l'architettura, oppure averlo gia' eseguito prima del meeting. Per una prova **solo meccanica** (stime non utilizzabili) aggiungi `--smoke`.

In [ ]:
!python -m pipeline.model.run

## 5. Il test: parameter recovery

Confronta le stime del modello con la verita' nascosta:
- **ROI stimato vs vero** per canale, con intervallo di credibilita' 90%;
- **copertura 90%**: quante volte il vero cade dentro l'intervallo (piu' alta = meglio);
- **MAE% sulle curve** di risposta nel range di spesa osservato (piu' basso = meglio).

Se ROI e curve sono vicini alla verita' con buona copertura, il modello "ci ha preso".

In [ ]:
!python -m pipeline.validation.recovery

## 6. (bonus) Allocazione del budget - canale + campagna

Lo stesso fit alimenta l'**allocator**: ripartizione ottima del budget per **canale** e **campagna**, con **grafici** (anche dentro l'Excel, foglio 'Grafici'). Esempio sui dati sintetici; in produzione coi tuoi budget e vincoli (`--budget`, `--min`, `--max`).

In [ ]:
!python -m pipeline.allocator.run --budget 420000 --max google=120000 --quarter-start 2026-07-06

## 7. Scarica i risultati (fit + recovery + allocazione)

Lo zip include `risultati.xlsx` (fogli Recovery, Canali, Campagne, **Grafici**) e i grafici PNG.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('dimostrazione', 'zip', 'pipeline/data/output')
files.download('dimostrazione.zip')